# 🎯 Technique 83: Top-p Sampling (Nucleus Sampling)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/83_top_p_sampling.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  **Technique #:** 83  **Difficulty:** Intermediate

## 📋 Description

Top-p sampling, also known as Nucleus Sampling, is a decoding strategy that dynamically selects from the smallest set of tokens whose cumulative probability exceeds a threshold p. Unlike top-k sampling which uses a fixed number of tokens, top-p adapts to the probability distribution, selecting more tokens when the distribution is flat and fewer when it's peaked. This provides a more flexible approach to controlling output diversity.

**When to use:**
- Want dynamic vocabulary selection based on confidence
- Need to balance diversity with coherence
- Top-k produces too many or too few candidates
- Working with tasks requiring variable creativity
- Combining with temperature for fine-grained control

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                    TOP-P SAMPLING EXPLAINED                  │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  STEP 1: SORT TOKENS BY PROBABILITY                          │
│                                                              │
│  Token    Probability                                        │
│  ─────────────────────                                       │
│  "the"    0.35  ████████████████████                       │
│  "a"      0.25  ██████████████                               │
│  "an"     0.15  ████████                                     │
│  "this"   0.10  █████                                        │
│  "that"   0.08  ████                                         │
│  "one"    0.04  ██                                           │
│  "some"   0.02  █                                            │
│  ...      ...                                                │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  STEP 2: SELECT NUCLEUS (CUMULATIVE PROBABILITY ≥ p)         │
│                                                              │
│  For p = 0.9:                                                │
│                                                              │
│  "the" (0.35) + "a" (0.25) + "an" (0.15) +                 │
│  "this" (0.10) + "that" (0.08) = 0.93 ≥ 0.9 ✓              │
│                                                              │
│  Nucleus = {the, a, an, this, that}  ←  Sample from these  │
│  Excluded = {one, some, ...}         ←  Ignore these         │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│              TOP-P VS TOP-K COMPARISON                       │
└─────────────────────────────────────────────────────────────┘
                            │
        ┌───────────────────┴───────────────────┐
        ▼                                       ▼
┌──────────────┐                       ┌──────────────┐
│   TOP-K      │                       │   TOP-P      │
│  (Fixed k)   │                       │ (Dynamic set)│
└──────────────┘                       └──────────────┘
        │                                       │
        ▼                                       ▼
   Always selects                    Adapts to distribution
   k tokens                          shape
        │                                       │
        ▼                                       ▼
   Peaky dist: May include           Peaky dist: Few tokens
   unlikely tokens                   (focused)
        │                                       │
        ▼                                       ▼
   Flat dist: May exclude            Flat dist: Many tokens
   valid options                     (diverse)
```

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai matplotlib numpy

import openai
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict
from getpass import getpass

In [ ]:
# Configure API
openai.api_key = getpass("Enter your OpenAI API key: ")

## 🛠️ Implementation: Top-p Explorer

In [ ]:
class TopPExplorer:
    """Explore and compare different top-p settings."""
    
    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
    
    def generate(
        self, 
        prompt: str, 
        top_p: float,
        temperature: float = 0.7,
        n: int = 1,
        max_tokens: int = 150
    ) -> List[str]:
        """Generate responses at specified top_p."""
        responses = []
        
        for _ in range(n):
            response = openai.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens
            )
            responses.append(response.choices[0].message.content.strip())
        
        return responses
    
    def compare_top_p_values(
        self,
        prompt: str,
        top_p_values: List[float],
        temperature: float = 0.7,
        samples_per_setting: int = 3
    ) -> Dict[float, List[str]]:
        """Compare responses across different top-p values."""
        results = {}
        
        for top_p in top_p_values:
            print(f"Generating with top_p={top_p}...")
            results[top_p] = self.generate(
                prompt, 
                top_p,
                temperature=temperature,
                n=samples_per_setting
            )
        
        return results
    
    def visualize_nucleus_sampling(self):
        """Visualize how top-p affects token selection."""
        # Simulate a probability distribution
        np.random.seed(42)
        
        # Create two distributions: peaked and flat
        peaked_logits = np.array([3.0, 2.0, 1.0, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01])
        flat_logits = np.array([1.5, 1.4, 1.3, 1.2, 1.1, 1.0, 0.9, 0.8, 0.7, 0.6])
        
        def softmax(logits):
            exp_logits = np.exp(logits - np.max(logits))
            return exp_logits / exp_logits.sum()
        
        peaked_probs = softmax(peaked_logits)
        flat_probs = softmax(flat_logits)
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        top_p_values = [0.5, 0.8, 1.0]
        
        for i, (dist_name, probs) in enumerate([("Peaked", peaked_probs), ("Flat", flat_probs)]):
            sorted_indices = np.argsort(probs)[::-1]
            sorted_probs = probs[sorted_indices]
            cumsum = np.cumsum(sorted_probs)
            
            for j, top_p in enumerate(top_p_values):
                ax = axes[i, j]
                
                # Find nucleus
                nucleus_size = np.searchsorted(cumsum, top_p) + 1
                
                # Color nucleus tokens differently
                colors = ['green' if k < nucleus_size else 'gray' for k in range(len(probs))]
                
                ax.bar(range(len(probs)), sorted_probs, color=colors, alpha=0.7)
                ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
                ax.set_title(f"{dist_name} Dist - top_p={top_p}\nNucleus size: {nucleus_size} tokens")
                ax.set_xlabel("Token Rank")
                ax.set_ylabel("Probability")
        
        plt.tight_layout()
        plt.show()
        
        return fig

## 💡 Basic Example: Top-p Comparison

In [ ]:
# Initialize explorer
explorer = TopPExplorer(model="gpt-4o-mini")

# Test prompt
test_prompt = "Complete this sentence creatively: The future of AI is"

print("🎯 Top-p Sampling Comparison")
print("="*60)
print(f"\nPrompt: '{test_prompt}'\n")

# Compare different top-p values
top_p_values = [0.3, 0.7, 1.0]
results = explorer.compare_top_p_values(
    test_prompt, 
    top_p_values, 
    temperature=0.8,
    samples_per_setting=2
)

# Display results
for top_p in top_p_values:
    print(f"\n{'='*60}")
    print(f"top_p = {top_p}")
    print(f"{'='*60}")
    
    for i, response in enumerate(results[top_p], 1):
        print(f"\nSample {i}: {response}")

In [ ]:
# Visualize nucleus sampling
print("\n" + "="*60)
print("📊 VISUALIZING NUCLEUS SAMPLING")
print("="*60)
print("\nGreen bars = tokens in nucleus (selected for sampling)")
print("Gray bars = tokens excluded from sampling\n")

explorer.visualize_nucleus_sampling()

## 🌍 Real-World Example: Top-p for Different Tasks

In [ ]:
# Real-world: Task-specific top-p recommendations

tasks = [
    {
        "name": "Factual Q&A",
        "prompt": "What is the capital of France?",
        "recommended_top_p": 0.1,
        "recommended_temp": 0.1,
        "reason": "Factual answers need high confidence"
    },
    {
        "name": "Code Completion",
        "prompt": "Complete: def factorial(n):",
        "recommended_top_p": 0.5,
        "recommended_temp": 0.2,
        "reason": "Code needs structure but some flexibility"
    },
    {
        "name": "Story Continuation",
        "prompt": "Continue the story: Once upon a time...",
        "recommended_top_p": 0.9,
        "recommended_temp": 0.8,
        "reason": "Creative writing benefits from diversity"
    },
    {
        "name": "Brainstorming",
        "prompt": "List creative uses for a paperclip.",
        "recommended_top_p": 1.0,
        "recommended_temp": 0.9,
        "reason": "Maximum diversity for ideation"
    }
]

print("🎯 TASK-SPECIFIC TOP-P GUIDE")
print("="*60)

for task in tasks:
    print(f"\n{'='*60}")
    print(f"Task: {task['name']}")
    print(f"Recommended: top_p={task['recommended_top_p']}, temp={task['recommended_temp']}")
    print(f"Reason: {task['reason']}")
    print(f"\nPrompt: {task['prompt']}")
    
    # Generate sample
    response = explorer.generate(
        task['prompt'], 
        top_p=task['recommended_top_p'],
        temperature=task['recommended_temp'],
        max_tokens=50
    )[0]
    print(f"\nSample Output: {response}")

In [ ]:
# Compare top-p vs top-k for same task
print("\n" + "="*60)
print("📊 TOP-P VS TOP-K COMPARISON")
print("="*60)

comparison_prompt = "Write a creative product tagline for eco-friendly water bottles."

print(f"\nPrompt: {comparison_prompt}\n")

# Test different configurations
configs = [
    ("top_p=0.9", {"top_p": 0.9, "temperature": 0.8}),
    ("top_k equivalent*", {"top_p": 1.0, "temperature": 0.8}),  # *top_k not directly available
    ("top_p=0.5", {"top_p": 0.5, "temperature": 0.8}),
    ("top_p=1.0", {"top_p": 1.0, "temperature": 0.8}),
]

for name, params in configs:
    print(f"\n{'='*40}")
    print(f"Configuration: {name}")
    print(f"{'='*40}")
    
    for i in range(2):
        response = explorer.generate(
            comparison_prompt,
            top_p=params["top_p"],
            temperature=params["temperature"],
            max_tokens=30
        )[0]
        print(f"  {i+1}. {response}")

## ⚠️ Failure Case: Misconfigured Top-p

In [ ]:
print("⚠️ FAILURE CASE: Misconfigured Top-p\n")
print("="*60)

# Example 1: Too restrictive for creative task
print("\n❌ EXAMPLE 1: Too Restrictive (top_p=0.1 for creative writing)")
print("-"*60)

creative_prompt = "Write a poem about the ocean."

print("With top_p=0.1 (too restrictive):")
for i in range(3):
    response = explorer.generate(
        creative_prompt, 
        top_p=0.1, 
        temperature=0.9,
        max_tokens=50
    )[0]
    print(f"  {i+1}. {response[:60]}...")

print("\nWith top_p=0.9 (better):")
for i in range(3):
    response = explorer.generate(
        creative_prompt, 
        top_p=0.9, 
        temperature=0.9,
        max_tokens=50
    )[0]
    print(f"  {i+1}. {response[:60]}...")

# Example 2: Too permissive for factual task
print("\n\n❌ EXAMPLE 2: Too Permissive (top_p=1.0 for factual QA)")
print("-"*60)

factual_prompt = "What is 2+2? Answer with just the number."

print("With top_p=1.0 (too permissive):")
responses = explorer.generate(factual_prompt, top_p=1.0, temperature=0.9, n=5, max_tokens=10)
for i, r in enumerate(responses, 1):
    print(f"  {i}. '{r}'")

print("\nWith top_p=0.1 (focused):")
responses = explorer.generate(factual_prompt, top_p=0.1, temperature=0.1, n=5, max_tokens=10)
for i, r in enumerate(responses, 1):
    print(f"  {i}. '{r}'")

print("\n" + "="*60)
print("LESSONS LEARNED:")
print("="*60)
print("""
1. LOW TOP-P (0.1-0.3) FOR:
   - Factual questions
   - Classification tasks
   - Structured outputs
   - When confidence matters

2. MEDIUM TOP-P (0.5-0.8) FOR:
   - General conversation
   - Explanations
   - Balanced tasks

3. HIGH TOP-P (0.9-1.0) FOR:
   - Creative writing
   - Brainstorming
   - Exploring possibilities
   - When diversity matters

4. COMBINE WITH TEMPERATURE:
   - Low top_p + low temp = very focused
   - High top_p + high temp = very creative
   - Adjust both for fine control
""")

## 📊 Top-p Benchmarks

| Use Case | Top-p | Temperature | Effect |
|----------|-------|-------------|--------|
| Factual QA | 0.1-0.3 | 0.1-0.3 | Highly focused, consistent |
| Code Gen | 0.3-0.5 | 0.1-0.2 | Structured, syntactically correct |
| Classification | 0.1-0.2 | 0.1 | Deterministic labels |
| Summarization | 0.5-0.7 | 0.3-0.5 | Coherent, informative |
| Chat/Dialogue | 0.7-0.9 | 0.6-0.8 | Natural, varied |
| Creative Writing | 0.9-1.0 | 0.8-1.0 | Diverse, imaginative |
| Brainstorming | 1.0 | 0.9-1.2 | Maximum diversity |

**Default Recommendation:** top_p=1.0 (uses full vocabulary)

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
║              🎮 TOP-P INTERACTIVE PLAYGROUND                      ║
╚══════════════════════════════════════════════════════════════════╝

# Define your own prompt and test top-p values:

YOUR_PROMPT = """
[Your prompt here]
"""

YOUR_TOP_P_VALUES = [0.3, 0.7, 1.0]  # Modify as needed

YOUR_TEMPERATURE = 0.7

SAMPLES_PER_SETTING = 2

# Run comparison (uncomment to execute):
# my_explorer = TopPExplorer()
# my_results = my_explorer.compare_top_p_values(
#     YOUR_PROMPT, 
#     YOUR_TOP_P_VALUES, 
#     temperature=YOUR_TEMPERATURE,
#     samples_per_setting=SAMPLES_PER_SETTING
# )
# for top_p, responses in my_results.items():
#     print(f"\ntop_p={top_p}:")
#     for i, r in enumerate(responses, 1):
#         print(f"  {i}. {r[:100]}...")

## 💡 Tips & Tricks

### Top-p + Temperature Combinations

| Goal | Top-p | Temp | Use Case |
|------|-------|------|----------|
| Maximum determinism | 0.1 | 0.0 | Testing, debugging |
| Focused accuracy | 0.3 | 0.2 | Factual tasks |
| Balanced output | 0.9 | 0.7 | General use |
| Creative diversity | 1.0 | 1.0 | Brainstorming |
| Controlled creativity | 0.8 | 0.9 | Creative with structure |

### When to Adjust Top-p
- **Decrease top-p** when outputs are too random
- **Increase top-p** when outputs are too repetitive
- **Keep at 1.0** if unsure (full vocabulary)

### Top-p vs Top-k
- Top-p is generally preferred over top-k
- Top-p adapts to the probability distribution
- Top-k uses fixed number regardless of confidence
- Most APIs only expose top_p parameter

## 📚 References

1. [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751) - Holtzman et al. (Original Nucleus Sampling paper)
2. [OpenAI API Top-p Parameter](https://platform.openai.com/docs/api-reference/chat/create#chat-create-top_p)
3. [How to Generate Text](https://huggingface.co/blog/how-to-generate) - Hugging Face
4. [Sampling Techniques in Language Models](https://arxiv.org/abs/2210.14140) - Research survey
5. [Making the V in LLM-V Work](https://arxiv.org/abs/2305.07759) - Visualizing sampling effects